# 🧠 Accuracy and the Accuracy Paradox

Welcome to the hands-on explanation notebook for **Accuracy**! In this notebook, we will:
1. Define the components of binary classification: True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).
2. Implement an accuracy metric from scratch using NumPy and verify it against `scikit-learn`.
3. Demonstrate the **Accuracy Paradox** by simulating an imbalanced dataset (e.g., pipe leak detection where 99% of samples are negative).
4. Show how a useless "majority-class classifier" gets high accuracy but fails completely at the task.
5. Connect these concepts to why standard classification accuracy is not used in object detection systems like YOLO.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Calculating Accuracy from Scratch

Let's write a function to calculate TP, TN, FP, FN and combine them to find the accuracy.

In [ ]:
def calculate_outcomes(y_true, y_pred):
    """
    Calculate True Positives, True Negatives, False Positives, and False Negatives.
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, TN, FP, FN

def custom_accuracy(y_true, y_pred):
    TP, TN, FP, FN = calculate_outcomes(y_true, y_pred)
    total = TP + TN + FP + FN
    if total == 0:
        return 0.0
    return (TP + TN) / total

# Test arrays (e.g., bounding box classification checks)
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

TP, TN, FP, FN = calculate_outcomes(y_true, y_pred)
acc_scratch = custom_accuracy(y_true, y_pred)
acc_sklearn = accuracy_score(y_true, y_pred)

print("--- Outcome Stats ---")
print(f"TP: {TP} | TN: {TN} | FP: {FP} | FN: {FN}")
print(f"Custom Accuracy: {acc_scratch:.4f}")
print(f"Sklearn Accuracy: {acc_sklearn:.4f}")

## 2. The Accuracy Paradox (Class Imbalance)

To understand why accuracy can be misleading, let's simulate the gas pipe leak detection dataset:
-   Total samples = 1,000 pipes.
-   Leaks (Class 1) = 10 (1% of the dataset).
-   Normal (Class 0) = 990 (99% of the dataset).

Let's test two classifiers:
1.  **Dumb Classifier:** Always predicts "Normal" (Class 0) regardless of the image.
2.  **Sensible Classifier:** A real machine learning model that makes some correct and incorrect predictions.

In [ ]:
# Create imbalanced ground truth labels
y_true_imbalanced = np.zeros(1000)
# Inject 10 leaks (Class 1) at random indexes
leak_indices = np.random.choice(1000, 10, replace=False)
y_true_imbalanced[leak_indices] = 1

# 1. Dumb Classifier Predictions (All 0s)
y_pred_dumb = np.zeros(1000)

# 2. Sensible Classifier Predictions
y_pred_sensible = np.zeros(1000)
# Correctly identify 8 leaks
correct_leaks_detected = leak_indices[:8]
y_pred_sensible[correct_leaks_detected] = 1
# Generate 15 false positives at random non-leak indexes
non_leak_indices = np.where(y_true_imbalanced == 0)[0]
false_alarms = np.random.choice(non_leak_indices, 15, replace=False)
y_pred_sensible[false_alarms] = 1

# Calculate metrics
acc_dumb = custom_accuracy(y_true_imbalanced, y_pred_dumb)
acc_sensible = custom_accuracy(y_true_imbalanced, y_pred_sensible)

tp_dumb, tn_dumb, fp_dumb, fn_dumb = calculate_outcomes(y_true_imbalanced, y_pred_dumb)
tp_sensible, tn_sensible, fp_sensible, fn_sensible = calculate_outcomes(y_true_imbalanced, y_pred_sensible)

# Display comparisons
df_compare = pd.DataFrame({
    'Metric': ['Accuracy', 'True Positives (Leaks Found)', 'False Negatives (Missed Leaks)', 'False Positives (False Alarms)'],
    'Dumb Model': [f"{acc_dumb*100:.2f}%", tp_dumb, fn_dumb, fp_dumb],
    'Sensible Model': [f"{acc_sensible*100:.2f}%", tp_sensible, fn_sensible, fp_sensible]
})

print(df_compare.to_string(index=False))

Look at the results!
-   The **Dumb Model** gets **99.00% accuracy**, but fails to find a single leak (0 True Positives, 10 Missed Leaks).
-   The **Sensible Model** gets a slightly lower **97.70% accuracy**, but it successfully finds 8 out of 10 leaks (8 True Positives, only 2 Missed Leaks).
This is the **Accuracy Paradox**: the model with lower accuracy is actually the one we want to deploy!

In [ ]:
# Visualize comparison
plt.figure(figsize=(10, 5))
x = np.arange(2)
width = 0.35

plt.bar(x - width/2, [tp_dumb, tp_sensible], width, label='True Leaks Detected (TP)', color='green')
plt.bar(x + width/2, [fn_dumb, fn_sensible], width, label='Missed Leaks (FN)', color='red')

plt.xticks(x, ['Dumb Model', 'Sensible Model'])
plt.ylabel('Count')
plt.title('Leak Detection Comparison: Dumb vs. Sensible Model')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 💡 Connection to Computer Vision & YOLO
Why don't we use standard classification accuracy to evaluate object detectors like YOLO?
1.  **Infinite True Negatives:** In an image, there are an infinite number of bounding boxes that *do not* contain an object. If a model predicts 2 bounding boxes correctly, how many "True Negatives" did it make? It correctly ignored millions of other possible boxes. If we plug $TN = \infty$ into the accuracy formula:
    $$\text{Accuracy} = \frac{TP + \infty}{TP + \infty + FP + FN} = 1.0 = 100\%$$
    Accuracy is always 100%, making it completely useless for object detection!
2.  **Evaluating Localization:** Accuracy cannot measure how tightly a bounding box overlaps with the ground truth. To solve this, computer vision uses **Intersection over Union (IoU)** to evaluate localization, and **mean Average Precision (mAP)** (which ignores True Negatives) to evaluate classification.